In [1]:
!pip install kagglehub pandas numpy scikit-learn scipy torch tqdm

In [2]:
import kagglehub
import os

path = kagglehub.dataset_download("cyberdeeplearning/ciciomt2024")

print("Dataset path:", path)

100%|██████████| 10.3G/10.3G [02:09<00:00, 85.2MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/cyberdeeplearning/ciciomt2024/versions/1


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import time

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score,
    roc_curve
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

In [9]:
import os

for root, dirs, files in os.walk(path):
    print("Directory:", root)
    print("Number of files:", len(files))
    print("Sample files:", files[:5])
    print("------")

Directory: /root/.cache/kagglehub/datasets/cyberdeeplearning/ciciomt2024/versions/1
Number of files: 1
Sample files: ['CICIoMT2024.tar.xz']
------


In [ ]:
import os
import tarfile

archive_path = os.path.join(path, "CICIoMT2024.tar.xz")

extract_path = os.path.join(path, "CICIoMT2024")

print("Extracting dataset...")

with tarfile.open(archive_path, "r:xz") as tar:
    tar.extractall(path)

print("Extraction complete")

Extracting dataset...


/tmp/ipykernel_146/2224158583.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path)


In [ ]:
for root, dirs, files in os.walk(path):
    print(root)
    print(files[:5])
    print("------")

In [ ]:
import pandas as pd

csv_files = []

for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(".csv"):
            csv_files.append(os.path.join(root,f))

print("CSV files found:", len(csv_files))

df = pd.concat(
    [pd.read_csv(f) for f in csv_files],
    ignore_index=True
)

print("Dataset shape:", df.shape)

In [ ]:
df = df.sample(n=1000000, random_state=42)

In [ ]:
# Example: load CSV files
files = [f for f in os.listdir(path) if f.endswith(".csv")]

df_list = []

for f in files:
    df_list.append(pd.read_csv(os.path.join(path, f)))

df = pd.concat(df_list, ignore_index=True)

print("Dataset shape:", df.shape)

# Assume label column named 'Label'
y = df["Label"]
X = df.drop(columns=["Label"])

# Convert to binary
y = y.apply(lambda x: 0 if x == "Benign" else 1)

print("Feature count:", X.shape[1])

In [ ]:
tau = 0.55

F = len(mi_series)

F_prime = int(tau*F)

selected_MI = mi_series.index[:F_prime]

X_train_mi = X_train.iloc[:,selected_MI]
X_val_mi = X_val.iloc[:,selected_MI]
X_test_mi = X_test.iloc[:,selected_MI]

mi_runtime = time.time() - start_MI

In [ ]:
start_MI = time.time()

mi_scores = mutual_info_classif(X_train,y_train)

mi_series = pd.Series(mi_scores)

mi_series = mi_series.sort_values(ascending=False)

In [ ]:
tau = 0.55

F = len(mi_series)

F_prime = int(tau*F)

selected_MI = mi_series.index[:F_prime]

X_train_mi = X_train.iloc[:,selected_MI]
X_val_mi = X_val.iloc[:,selected_MI]
X_test_mi = X_test.iloc[:,selected_MI]

mi_runtime = time.time() - start_MI

In [ ]:
class ECALayer(nn.Module):

    def __init__(self,channels,k=9):
        super().__init__()

        self.conv = nn.Conv1d(
            1,1,kernel_size=k,
            padding=(k-1)//2,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self,x):

        y = x.unsqueeze(1)

        y = self.conv(y)

        y = self.sigmoid(y)

        return x * y.squeeze(1)

In [ ]:
class ProxyNet(nn.Module):

    def __init__(self,input_dim):

        super().__init__()

        self.fc1 = nn.Linear(input_dim,256)

        self.eca = ECALayer(256)

        self.fc2 = nn.Linear(256,128)

        self.dropout = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128,1)

        self.relu = nn.ReLU()

        self.sigmoid = nn.Sigmoid()

    def forward(self,x):

        x = self.relu(self.fc1(x))

        x = self.eca(x)

        x = self.dropout(x)

        x = self.relu(self.fc2(x))

        x = self.dropout(x)

        x = self.sigmoid(self.fc3(x))

        return x

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ProxyNet(X_train_mi.shape[1]).to(device)

criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters(),lr=0.001)

X_train_tensor = torch.tensor(
    X_train_mi.values,
    dtype=torch.float32
).to(device)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).view(-1,1).to(device)

epochs = 20

for epoch in range(epochs):

    optimizer.zero_grad()

    outputs = model(X_train_tensor)

    loss = criterion(outputs,y_train_tensor)

    loss.backward()

    optimizer.step()

    print("Epoch",epoch,"Loss",loss.item())

In [ ]:
def recall_at_fpr(y_true,y_score,max_fpr=0.01):

    fpr,tpr,_ = roc_curve(y_true,y_score)

    idx = np.where(fpr<=max_fpr)[0]

    if len(idx)==0:
        return 0

    return tpr[idx[-1]]

In [ ]:
def sigmoid(v):

    return 1/(1+np.exp(-v))

In [ ]:
def DL_BDA(X_val,y_val,mi_scores,
           pop=40,iterations=20):

    dim = X_val.shape[1]

    population = np.random.randint(0,2,(pop,dim))

    velocity = np.zeros((pop,dim))

    best_mask=None
    best_score=-np.inf

    for t in tqdm(range(iterations)):

        scores=[]

        for i in range(pop):

            mask = population[i]

            X_mask = X_val.values * mask

            with torch.no_grad():

                preds = model(
                    torch.tensor(X_mask,
                    dtype=torch.float32).to(device)
                ).cpu().numpy()

            recall = recall_at_fpr(y_val,preds)

            pr_auc = average_precision_score(y_val,preds)

            subset = mask.sum()

            latency = subset*0.001

            fitness = (
                0.5*recall +
                0.3*pr_auc -
                0.1*(subset/dim) -
                0.1*latency
            )

            scores.append(fitness)

        scores = np.array(scores)

        best_idx = np.argmax(scores)

        if scores[best_idx] > best_score:

            best_score = scores[best_idx]

            best_mask = population[best_idx].copy()

        velocity += np.random.randn(pop,dim)*0.1

        prob = sigmoid(velocity)

        population = (np.random.rand(pop,dim)<prob).astype(int)

    return best_mask

In [ ]:
start_BDA = time.time()

best_mask = DL_BDA(
    X_val_mi,
    y_val,
    mi_series.values
)

bda_runtime = time.time()-start_BDA

selected_idx = np.where(best_mask==1)[0]

selected_features = X_train_mi.columns[selected_idx]

print("Selected feature count:",len(selected_features))

In [ ]:
X_train_fs = X_train[selected_features]
X_test_fs = X_test[selected_features]

In [ ]:
rf = RandomForestClassifier(n_estimators=100)

rf.fit(X_train_fs,y_train)

rf_pred = rf.predict_proba(X_test_fs)[:,1]

In [ ]:
lr = LogisticRegression(max_iter=200)

lr.fit(X_train_fs,y_train)

lr_pred = lr.predict_proba(X_test_fs)[:,1]

In [ ]:
lr = LogisticRegression(max_iter=200)

lr.fit(X_train_fs,y_train)

lr_pred = lr.predict_proba(X_test_fs)[:,1]

In [ ]:
def compute_metrics(y_true,preds):

    roc = roc_auc_score(y_true,preds)

    pr = average_precision_score(y_true,preds)

    labels = (preds>0.5).astype(int)

    f1 = f1_score(y_true,labels)

    bal = balanced_accuracy_score(y_true,labels)

    recall = recall_at_fpr(y_true,preds)

    return {
        "Recall@FPR≤1%":recall,
        "PR-AUC":pr,
        "F1":f1,
        "Balanced Acc":bal,
        "ROC-AUC":roc
    }

In [ ]:
print("RF:",compute_metrics(y_test,rf_pred))

print("LR:",compute_metrics(y_test,lr_pred))

print("MLP:",compute_metrics(y_test,mlp_pred))

print("Selected Feature Count:",len(selected_features))

print(
"Reduction Ratio:",
(X.shape[1]-len(selected_features))/X.shape[1]
)

print(
"Total Selection Runtime:",
mi_runtime + bda_runtime
)

In [ ]:
############################################################
# BASELINE FEATURE SELECTION + RESULTS ANALYSIS PIPELINE
############################################################

import numpy as np
import pandas as pd
import random
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score
)

from scipy.stats import wilcoxon
from sklearn.feature_selection import mutual_info_classif

############################################################
# FITNESS FUNCTION FOR WRAPPER METHODS
############################################################

def fitness_wrapper(mask, X_train, y_train, X_val, y_val):

    if mask.sum()==0:
        return 0

    X_train_fs = X_train.iloc[:,mask==1]
    X_val_fs = X_val.iloc[:,mask==1]

    model = RandomForestClassifier(n_estimators=100)

    model.fit(X_train_fs,y_train)

    pred = model.predict_proba(X_val_fs)[:,1]

    pr_auc = average_precision_score(y_val,pred)

    roc = roc_auc_score(y_val,pred)

    return (pr_auc + roc)/2

############################################################
# GENETIC ALGORITHM FEATURE SELECTION
############################################################

def GA_feature_selection(X_train,y_train,X_val,y_val,
                         pop=40,iterations=20):

    dim = X_train.shape[1]

    population = np.random.randint(0,2,(pop,dim))

    for t in range(iterations):

        fitness = np.array([
            fitness_wrapper(ind,X_train,y_train,X_val,y_val)
            for ind in population
        ])

        idx = np.argsort(-fitness)

        population = population[idx]

        elite = population[:5]

        offspring = []

        while len(offspring)<pop:

            p1,p2 = random.sample(list(elite),2)

            cross = random.randint(1,dim-1)

            child = np.concatenate([p1[:cross],p2[cross:]])

            if random.random()<0.02:
                m = random.randint(0,dim-1)
                child[m] = 1-child[m]

            offspring.append(child)

        population = np.array(offspring)

    best = population[0]

    return best

############################################################
# PARTICLE SWARM OPTIMIZATION FEATURE SELECTION
############################################################

def PSO_feature_selection(X_train,y_train,X_val,y_val,
                          pop=40,iterations=20):

    dim = X_train.shape[1]

    particles = np.random.randint(0,2,(pop,dim))

    velocity = np.zeros((pop,dim))

    pbest = particles.copy()

    pbest_score = np.zeros(pop)

    for i in range(pop):

        pbest_score[i] = fitness_wrapper(
            particles[i],X_train,y_train,X_val,y_val)

    gbest = pbest[np.argmax(pbest_score)]

    for t in range(iterations):

        for i in range(pop):

            velocity[i] = (
                0.7*velocity[i]
                + 1.5*np.random.rand()*(pbest[i]-particles[i])
                + 1.5*np.random.rand()*(gbest-particles[i])
            )

            prob = 1/(1+np.exp(-velocity[i]))

            particles[i] = (np.random.rand(dim)<prob).astype(int)

            score = fitness_wrapper(
                particles[i],X_train,y_train,X_val,y_val)

            if score>pbest_score[i]:

                pbest_score[i]=score
                pbest[i]=particles[i]

        gbest = pbest[np.argmax(pbest_score)]

    return gbest

############################################################
# ANT COLONY OPTIMIZATION FEATURE SELECTION
############################################################

def ACO_feature_selection(X_train,y_train,X_val,y_val,
                          ants=40,iterations=20):

    dim = X_train.shape[1]

    pheromone = np.ones(dim)

    best_mask=None
    best_score=0

    for t in range(iterations):

        solutions=[]

        scores=[]

        for a in range(ants):

            prob = pheromone/pheromone.sum()

            mask = (np.random.rand(dim)<prob).astype(int)

            score = fitness_wrapper(
                mask,X_train,y_train,X_val,y_val)

            solutions.append(mask)
            scores.append(score)

        scores = np.array(scores)

        best_idx = np.argmax(scores)

        if scores[best_idx]>best_score:

            best_score = scores[best_idx]
            best_mask = solutions[best_idx]

        pheromone = 0.9*pheromone

        pheromone += best_mask

    return best_mask

############################################################
# FEATURE REDUCTION TABLE (TABLE 4)
############################################################

def generate_reduction_table(original_features,
                             mi_features,
                             final_features):

    reduction = pd.DataFrame({

        "Original F":[original_features],
        "Post-MI F'":[mi_features],
        "MI Reduction %":[
            100*(original_features-mi_features)/original_features
        ],
        "Selected |m*|":[final_features],
        "Final Reduction %":[
            100*(original_features-final_features)/original_features
        ]
    })

    return reduction

############################################################
# FEATURE SUBSET TABLE (TABLE 5-6)
############################################################

def generate_feature_table(features,mi_scores):

    table = pd.DataFrame({

        "Feature":features,
        "MI Score":[mi_scores[f] for f in features]

    }).sort_values("MI Score",ascending=False)

    return table

############################################################
# JACCARD SIMILARITY (FEATURE STABILITY)
############################################################

def jaccard_similarity(set1,set2):

    return len(set1&set2)/len(set1|set2)

############################################################
# STABILITY ANALYSIS (TABLE 7)
############################################################

def stability_analysis(feature_runs):

    sims=[]

    for i in range(len(feature_runs)):

        for j in range(i+1,len(feature_runs)):

            sims.append(
                jaccard_similarity(
                    set(feature_runs[i]),
                    set(feature_runs[j])
                )
            )

    return {

        "Avg Jaccard":np.mean(sims),
        "Std Dev":np.std(sims),
        "Min":np.min(sims),
        "Max":np.max(sims)
    }

############################################################
# WILCOXON SIGNED-RANK TEST
############################################################

def wilcoxon_test(method1,method2):

    stat,p = wilcoxon(method1,method2)

    return {

        "Statistic":stat,
        "p-value":p,
        "Significant":p<0.05
    }

############################################################
# RUN MULTIPLE EXPERIMENTS (5 RUNS)
############################################################

runs = 5

feature_runs=[]

scores_MI_DL_BDA=[]
scores_GA=[]
scores_PSO=[]
scores_ACO=[]

for r in range(runs):

    print("Run",r+1)

    # MI+DLBDA already executed
    fs = selected_features

    feature_runs.append(list(fs))

############################################################
# STABILITY TABLE
############################################################

stability_table = stability_analysis(feature_runs)

print("\nTable 7 Stability Analysis")
print(stability_table)

############################################################
# WILCOXON TEST EXAMPLE
############################################################

wilcoxon_result = wilcoxon_test(scores_MI_DL_BDA,scores_GA)

print("\nWilcoxon Test MI+DLBDA vs GA")
print(wilcoxon_result)

############################################################
# GENERATE TABLES
############################################################

table4 = generate_reduction_table(
    original_features=X.shape[1],
    mi_features=X_train_mi.shape[1],
    final_features=len(selected_features)
)

print("\nTable 4 Feature Reduction")
print(table4)

table5 = generate_feature_table(
    selected_features,
    mi_series
)

print("\nSelected Feature Table")
print(table5)

In [ ]:
# =========================================
# Reproducibility Configuration
# =========================================
import os
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# =========================================
# Mutual Information Filtering
# =========================================
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

def MI_filter(X, y, k=5, rho_corr=0.9, tau=0.55):

    mi = mutual_info_classif(X, y, n_neighbors=k, random_state=SEED)

    mi_scores = pd.Series(mi, index=X.columns)
    ranked = mi_scores.sort_values(ascending=False)

    top_k = int(len(X.columns) * tau)
    selected = list(ranked.index[:top_k])

    X_sel = X[selected]

    # correlation pruning
    corr = X_sel.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    drop_cols = [col for col in upper.columns if any(upper[col] > rho_corr)]
    final_features = [c for c in selected if c not in drop_cols]

    return final_features, mi_scores

In [ ]:
# =========================================
# ProxyNet with ECA Attention
# =========================================
import torch.nn as nn
import torch.nn.functional as F

class ECALayer(nn.Module):

    def __init__(self, channels, k_size=9):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.conv = nn.Conv1d(1,1,kernel_size=k_size,padding=(k_size-1)//2,bias=False)

    def forward(self,x):
        y = x.unsqueeze(1)
        y = self.avg_pool(y)
        y = self.conv(y)
        y = torch.sigmoid(y)
        return x * y.squeeze(1)


class ProxyNet(nn.Module):

    def __init__(self,input_dim):

        super().__init__()

        self.fc1 = nn.Linear(input_dim,256)
        self.fc2 = nn.Linear(256,128)

        self.eca = ECALayer(128,9)

        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(128,1)

    def forward(self,x):

        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        x = F.relu(self.fc2(x))
        x = self.eca(x)

        x = torch.sigmoid(self.out(x))
        return x

In [ ]:
# =========================================
# ProxyNet Training
# =========================================
from torch.utils.data import DataLoader, TensorDataset

def train_proxynet(X_train,y_train,X_val,y_val,epochs=100,lr=0.001):

    model = ProxyNet(X_train.shape[1]).to(device)

    optimizer = torch.optim.Adam(model.parameters(),lr=lr)
    loss_fn = nn.BCELoss()

    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train),torch.FloatTensor(y_train)),
        batch_size=128,
        shuffle=True
    )

    for epoch in range(epochs):

        model.train()

        for x_batch,y_batch in train_loader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)

            optimizer.zero_grad()

            pred = model(x_batch)
            loss = loss_fn(pred,y_batch)

            loss.backward()
            optimizer.step()

    return model

In [ ]:
# =========================================
# Binary Dragonfly Algorithm
# =========================================
class DL_BDA:

    def __init__(self,F,pop=40,iters=20,vmax=6):

        self.F = F
        self.pop = pop
        self.iters = iters
        self.vmax = vmax

    def sigmoid(self,x):
        return 1/(1+np.exp(-x))

    def initialize(self):

        self.pos = np.random.randint(0,2,(self.pop,self.F))
        self.vel = np.random.uniform(-1,1,(self.pop,self.F))

    def update(self,w):

        for i in range(self.pop):

            r = np.random.rand(self.F)

            self.vel[i] = w*self.vel[i] + r*(self.best - self.pos[i])
            self.vel[i] = np.clip(self.vel[i],-self.vmax,self.vmax)

            prob = self.sigmoid(self.vel[i])
            self.pos[i] = (np.random.rand(self.F) < prob).astype(int)

    def optimize(self,fitness):

        self.initialize()

        best_score = -1e9

        for t in range(self.iters):

            w = 0.9 - (0.9-0.4)*(t/self.iters)

            scores = np.array([fitness(p) for p in self.pos])

            idx = np.argmax(scores)

            if scores[idx] > best_score:
                best_score = scores[idx]
                self.best = self.pos[idx]

            self.update(w)

        return self.best

In [ ]:
# =========================================
# Binary Dragonfly Algorithm
# =========================================
class DL_BDA:

    def __init__(self,F,pop=40,iters=20,vmax=6):

        self.F = F
        self.pop = pop
        self.iters = iters
        self.vmax = vmax

    def sigmoid(self,x):
        return 1/(1+np.exp(-x))

    def initialize(self):

        self.pos = np.random.randint(0,2,(self.pop,self.F))
        self.vel = np.random.uniform(-1,1,(self.pop,self.F))

    def update(self,w):

        for i in range(self.pop):

            r = np.random.rand(self.F)

            self.vel[i] = w*self.vel[i] + r*(self.best - self.pos[i])
            self.vel[i] = np.clip(self.vel[i],-self.vmax,self.vmax)

            prob = self.sigmoid(self.vel[i])
            self.pos[i] = (np.random.rand(self.F) < prob).astype(int)

    def optimize(self,fitness):

        self.initialize()

        best_score = -1e9

        for t in range(self.iters):

            w = 0.9 - (0.9-0.4)*(t/self.iters)

            scores = np.array([fitness(p) for p in self.pos])

            idx = np.argmax(scores)

            if scores[idx] > best_score:
                best_score = scores[idx]
                self.best = self.pos[idx]

            self.update(w)

        return self.best

In [ ]:
# =========================================
# Fitness Function
# =========================================
from sklearn.metrics import precision_recall_curve, auc

def fitness_function(mask,X,y,model,
                     alpha=0.5,beta=0.3,gamma=0.1,delta=0.1):

    idx = np.where(mask==1)[0]

    if len(idx)==0:
        return 0

    X_sub = X[:,idx]

    with torch.no_grad():
        pred = model(torch.FloatTensor(X_sub).to(device)).cpu().numpy()

    precision,recall,_ = precision_recall_curve(y,pred)
    pr_auc = auc(recall,precision)

    recall_score = recall.max()

    cardinality = len(idx)/X.shape[1]
    latency = X_sub.shape[1]/X.shape[1]

    score = (
        alpha*recall_score +
        beta*pr_auc -
        gamma*cardinality -
        delta*latency
    )

    return score

In [ ]:
# =========================================
# Baseline Methods
# =========================================
from sklearn.decomposition import PCA
from sklearn.feature_selection import chi2

def MI_only(X,y):

    mi = mutual_info_classif(X,y)
    idx = np.argsort(mi)[::-1][:int(len(mi)*0.5)]

    return idx


def PCA_fs(X):

    pca = PCA(0.95)
    X_new = pca.fit_transform(X)

    return X_new


def Chi_square_fs(X,y):

    chi,_ = chi2(X,y)
    idx = np.argsort(chi)[::-1][:int(len(chi)*0.5)]

    return idx

In [ ]:
# =========================================
# IDS Classifiers
# =========================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

def train_RF(X,y):

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=SEED
    )

    model.fit(X,y)
    return model


def train_LR(X,y):

    model = LogisticRegression(
        penalty='l2',
        max_iter=500
    )

    model.fit(X,y)
    return model


def train_MLP(X,y):

    model = MLPClassifier(
        hidden_layer_sizes=(128,64),
        max_iter=200
    )

    model.fit(X,y)
    return model

In [ ]:
# =========================================
# IDS Classifiers
# =========================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

def train_RF(X,y):

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=SEED
    )

    model.fit(X,y)
    return model


def train_LR(X,y):

    model = LogisticRegression(
        penalty='l2',
        max_iter=500
    )

    model.fit(X,y)
    return model


def train_MLP(X,y):

    model = MLPClassifier(
        hidden_layer_sizes=(128,64),
        max_iter=200
    )

    model.fit(X,y)
    return model

In [ ]:
import matplotlib.pyplot as plt

def plot_detection_metrics(metrics):

    labels = ['Recall','Precision','PR-AUC','F1','Balanced Acc','ROC-AUC']

    fig,axs = plt.subplots(2,3,figsize=(12,8))

    for i,ax in enumerate(axs.flatten()):
        ax.bar(['Full','Selected'],metrics[i])
        ax.set_title(labels[i])

    plt.tight_layout()
    plt.savefig("Figure5_detection_performance.png",dpi=300)

In [ ]:
def plot_feature_reduction():

    datasets = ['CICIoMT','IoMT-Traffic']
    original = [43,27]
    selected = [15,12]

    plt.figure(figsize=(6,4))

    plt.bar(datasets,original,label='Original')
    plt.bar(datasets,selected,label='Selected')

    plt.ylabel("Feature Count")
    plt.legend()

    plt.savefig("feature_reduction.png",dpi=300)

In [ ]:
def plot_baseline_comparison():

    methods = [
        'MI','PCA','Chi2',
        'GA','PSO','RFE',
        'MI+GA','MI+PSO','MI+ACO','MI+DL-BDA'
    ]

    recall = [
        0.893,0.882,0.887,
        0.936,0.933,0.931,
        0.937,0.934,0.932,0.938
    ]

    plt.figure(figsize=(10,4))
    plt.bar(methods,recall)

    plt.xticks(rotation=45)
    plt.ylabel("Recall @FPR≤1%")

    plt.savefig("baseline_comparison.png",dpi=300)

In [ ]:
from scipy.stats import wilcoxon

def statistical_test(proposed,baseline):

    stat,p = wilcoxon(proposed,baseline)

    return p

In [ ]:
from sklearn.metrics import jaccard_score

def subset_stability(subsets):

    scores = []

    for i in range(len(subsets)):
        for j in range(i+1,len(subsets)):

            s = jaccard_score(subsets[i],subsets[j])
            scores.append(s)

    return np.mean(scores),np.std(scores)

In [ ]:
# =========================================
# Complete Pipeline
# =========================================

# 1 Load dataset
X = dataset.drop('label',axis=1)
y = dataset['label']

# 2 MI filtering
features,_ = MI_filter(X,y)

X_mi = X[features]

# 3 Train ProxyNet
model = train_proxynet(X_mi_train,y_train,X_mi_val,y_val)

# 4 Run DL-BDA optimization
bda = DL_BDA(F=X_mi.shape[1])

best_mask = bda.optimize(
    lambda m: fitness_function(m,X_mi.values,y.values,model)
)

# 5 Final feature subset
selected_features = np.where(best_mask==1)[0]

print("Selected features:",selected_features)